$${\color{yellow}{\text{Deep Learning Webinar-3, Saturday, May 17, 2025}}}$$



---

Load essential libraries

---

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
plt.style.use('dark_background')
%matplotlib inline
import sys
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
torch.set_printoptions(precision = 3, sci_mode = False)

---

Mount Google Drive folder if running Google Colab

---

In [ ]:
## Mount Google drive folder if running in Colab
if('google.colab' in sys.modules):
    from google.colab import drive
    drive.mount('/content/drive', force_remount = True)
    DIR = '/content/drive/MyDrive/Colab Notebooks/MAHE/Office of Online Education/MDS6304_Webinar_April2025'
    DATA_DIR = DIR+'/Data/'
else:
    DATA_DIR = 'Data/'

---

Matrix-matrix product using patient data matrix and a weights matrix:

![patient dataset](https://1drv.ms/i/s!AjTcbXuSD3I3hspfrgklysOtJMOjaA?embed=1&width=800)

$$\mathbf{Z} = \mathbf{XW}.$$

---

In [ ]:
# Patients data matrix
X = torch.tensor([[72, 120, 37.3, 104, 32.5],
                 [85, 130, 37.0, 110, 14],
                 [68, 110, 38.5, 125, 34],
                 [90, 140, 38.0, 130, 26],
                 [84, 132, 38.3, 146, 30],
                 [78, 128, 37.2, 102, 12]])
print(f'Patient data matrix X:\n {X}')

# Weights matrix
W = torch.tensor([[-0.1, 0.5, 0.3],
                  [0.9, 0.3, 0.5],
                  [-1.5, 0.4, 0.1],
                  [0.1, 0.1, -1.0],
                  [-1.2, 0.5, 0.8]])
print(f'Weights matrix:\n {W}')

# Raw scores matrix (matrix-matrix multiplication)
Z = torch.matmul(X, W)
print(f'Raw zcores matrix:\n {Z}')
# The raw scores are also referred to as the logits

Patient data matrix X:
 tensor([[ 72.0000, 120.0000,  37.3000, 104.0000,  32.5000],
        [ 85.0000, 130.0000,  37.0000, 110.0000,  14.0000],
        [ 68.0000, 110.0000,  38.5000, 125.0000,  34.0000],
        [ 90.0000, 140.0000,  38.0000, 130.0000,  26.0000],
        [ 84.0000, 132.0000,  38.3000, 146.0000,  30.0000],
        [ 78.0000, 128.0000,  37.2000, 102.0000,  12.0000]])
Weights matrix:
 tensor([[-0.1000,  0.5000,  0.3000],
        [ 0.9000,  0.3000,  0.5000],
        [-1.5000,  0.4000,  0.1000],
        [ 0.1000,  0.1000, -1.0000],
        [-1.2000,  0.5000,  0.8000]])
Raw zcores matrix:
 tensor([[ 16.2500, 113.5700,   7.3300],
        [ 47.2000, 114.3000,  -4.6000],
        [  6.1500, 111.9000, -18.5500],
        [ 41.8000, 128.2000,  -8.4000],
        [ 31.5500, 126.5200, -26.9700],
        [ 47.4000, 108.4800,  -1.2800]])


---

**Version-1** view of the matrix-matrix product $\mathbf{Z} = \mathbf{XW}$:

*What a particular neuron understands about a particular patient.*

![matrix-matrix product version-1](https://1drv.ms/i/c/37720f927b6ddc34/IQQdAOCwtndURKA-h4yvpTqlAYjBjlcweRSeMYkPvf7dwmQ?width=660)

$$\begin{align*}[\mathbf{Z}]_{i,j} &= (i,j)\text{-th element of }\mathbf{Z}\\&=\text{what the }j\text{th neuron learns about the } i\text{th patient}\\&=\mathbf{x}^{(i)}\cdot\mathbf{w}_j\\& = {\mathbf{x}^{(i)}}^\mathrm{T}\mathbf{w}_j\\\Rightarrow \underbrace{[\mathbf{Z}]_{{\color{yellow}0},{\color{cyan}2}}}_{{\color{yellow}0}\text{th patient},\,{\color{cyan}2}\text{nd neuron}} &= \mathbf{x}^{({\color{yellow}0})}\cdot\mathbf{w}_{{\color{cyan}2}}\\ &= \begin{bmatrix}72\\120\\37.3\\104\\32.5\end{bmatrix}\cdot\begin{bmatrix}0.3\\0.5\\0.1\\-1.0\\-0.8\end{bmatrix}\\ &= 7.33.\end{align*}$$

---

In [ ]:
## The (0, 2)-th element of the matrix-matrix product XW
Z = torch.matmul(X, W)
print(Z)
print(f'Element in (0,2)th position of Z = {Z[0, 2]}')
print(torch.dot(X[0, :], W[:, 2]))

tensor([[ 16.250, 113.570,   7.330],
        [ 47.200, 114.300,  -4.600],
        [  6.150, 111.900, -18.550],
        [ 41.800, 128.200,  -8.400],
        [ 31.550, 126.520, -26.970],
        [ 47.400, 108.480,  -1.280]])
Element in (0,2)th position of Z = 7.330000400543213
tensor(7.330)


---

**Version-2** view of the matrix-matrix product $\mathbf{Z} = \mathbf{XW}$:

*What a particular neuron understands about all the patients.*

![matrix-matrix product version-2](https://1drv.ms/i/c/37720f927b6ddc34/IQRm1-w-6TG0R4C4J4BizyzyAWIbcHzbEjgmx-0JFREdHsE?width=660)

$$\begin{align*}\mathbf{z}_j &= \mathbf{X}\mathbf{w}_j\\&=\text{what the } j\text{th neuron learns about the all the patients}\\&=w_{j,0}\times\textbf{HR}+w_{j,1}\times\textbf{BP}+w_{j,2}\times\textbf{Temp}+w_{j,3}\times\textbf{Sugar}+w_{j,4}\times\textbf{Vitamin D}\\&= w_{j,0}\mathbf{x}_0+w_{j,1}\mathbf{x}_1+w_{j,2}\mathbf{x}_2+w_{j,3}\mathbf{x}_3+w_{j,4}\mathbf{x}_4\\\Rightarrow\underbrace{\mathbf{z}_{{\color{cyan}0}}}_{{\color{cyan}0}\text{th neuron understanding}} &= \underbrace{\mathbf{X}}_{\color{yellow}{\text{all patients}}}\ \underbrace{\mathbf{w}_{{\color{cyan}0}}}_{{\color{cyan}0}\text{th neuron weights}}\\&= {\color{cyan}{-0.1}}\times\begin{bmatrix}{\color{yellow}{72}}\\{\color{yellow}{85}}\\{\color{yellow}{68}}\\{\color{yellow}{90}}\\{\color{yellow}{84}}\\{\color{yellow}{78}}\end{bmatrix}+{\color{cyan}{0.9}}\times\begin{bmatrix}{\color{yellow}{120}}\\{\color{yellow}{130}}\\{\color{yellow}{110}}\\{\color{yellow}{140}}\\{\color{yellow}{132}}\\{\color{yellow}{128}}\end{bmatrix}+({\color{cyan}{-1.5}})\times\begin{bmatrix}{\color{yellow}{37.3}}\\{\color{yellow}{37.0}}\\{\color{yellow}{38.5}}\\{\color{yellow}{38.0}}\\{\color{yellow}{38.3}}\\{\color{yellow}{37.2}}\end{bmatrix}+{\color{cyan}{0.1}}\times\begin{bmatrix}{\color{yellow}{104}}\\{\color{yellow}{110}}\\{\color{yellow}{125}}\\{\color{yellow}{130}}\\{\color{yellow}{146}}\\{\color{yellow}{102}}\end{bmatrix}+({\color{cyan}{-1.2}})\times\begin{bmatrix}{\color{yellow}{32.5}}\\{\color{yellow}{14}}\\{\color{yellow}{34}}\\{\color{yellow}{26}}\\{\color{yellow}{30}}\\{\color{yellow}{12}}\end{bmatrix}\\&=\begin{bmatrix}16.25\\47.20\\6.15\\41.80\\31.55\\47.40\end{bmatrix}.\end{align*}$$



---

In [ ]:
## The 0-th column of the matrix-matrix product XW
Z = torch.matmul(X, W)
print(Z)
print(f'0th coulmn of Z:\n {Z[:, 0]}')
print(torch.matmul(X, W[:, 0]))

tensor([[ 16.250, 113.570,   7.330],
        [ 47.200, 114.300,  -4.600],
        [  6.150, 111.900, -18.550],
        [ 41.800, 128.200,  -8.400],
        [ 31.550, 126.520, -26.970],
        [ 47.400, 108.480,  -1.280]])
0th coulmn of Z:
 tensor([16.250, 47.200,  6.150, 41.800, 31.550, 47.400])
tensor([16.250, 47.200,  6.150, 41.800, 31.550, 47.400])


---

**Version-3** view of the matrix-matrix product $\mathbf{Z} = \mathbf{XW}$:

*What all neurons understand about a particular patient.*

![matrix-matrix product version-3](https://1drv.ms/i/c/37720f927b6ddc34/IQRfO-qEJQ9mQYLH_f-lyjeQAaWV4FrDjTjaEHJpPB1PmCg?width=660)

$$\begin{align*}{\mathbf{z}^{(i)}}^\mathrm{T}&={\mathbf{x}^{(i)}}^\mathrm{T}\mathbf{W}\\&= \text{what is learned about the }i\text{th patient by all the neurons}\\&=i\text{th HR }\times{\mathbf{w}^{(0)}}^\mathrm{T}+i\text{th BP }\times{\mathbf{w}^{(1)}}^\mathrm{T}+i\text{th Temp }\times{\mathbf{w}^{(2)}}^\mathrm{T}+i\text{th Sugar }\times{\mathbf{w}^{(3)}}^\mathrm{T}+i\text{th Vitamin D }\times{\mathbf{w}^{(4)}}^\mathrm{T}\\&=x^{(i)}_0\times{\mathbf{w}^{(0)}}^\mathrm{T}+x^{(i)}_1\times{\mathbf{w}^{(1)}}^\mathrm{T}+x^{(i)}_2\times{\mathbf{w}^{(2)}}^\mathrm{T}+x^{(i)}_3\times{\mathbf{w}^{(3)}}^\mathrm{T}+x^{(i)}_4\times{\mathbf{w}^{(4)}}^\mathrm{T}\\\underbrace{\Rightarrow{{\mathbf{z}^{({\color{yellow}0})}}^\mathrm{T}}}_{{\color{yellow}{0}}\text{th patient understanding}}&=\underbrace{{{\mathbf{x}^{({\color{yellow}0})}}^\mathrm{T}}}_{{\color{yellow}{0}}\text{th patient}}\ \underbrace{\mathbf{W}}_{{\color{cyan}{\text{all neurons}}}}\\ &= {\color{yellow}{72}}\times\begin{bmatrix}{\color{cyan}{-0.1}} & {\color{cyan}{0.5}} & {\color{cyan}{0.3}}\end{bmatrix} \\&+ {\color{yellow}{120}}\times\begin{bmatrix}{\color{cyan}{0.9}} & {\color{cyan}{0.3}} & {\color{cyan}{0.5}}\end{bmatrix}\\&+{\color{yellow}{37.3}}\times\begin{bmatrix}{\color{cyan}{-1.5}} & {\color{cyan}{0.4}} & {\color{cyan}{0.1}}\end{bmatrix}\\&+{\color{yellow}{104}}\times\begin{bmatrix}{\color{cyan}{0.1}} & {\color{cyan}{0.1}} & {\color{cyan}{-1.0}}\end{bmatrix}\\&+{\color{yellow}{32.5}}\times\begin{bmatrix}{\color{cyan}{-1.2}} & {\color{cyan}{0.5}} & {\color{cyan}{-0.8}}\end{bmatrix}\\&=\begin{bmatrix}16.25 & 113.57 & 7.33\end{bmatrix}.\end{align*}$$


---

In [ ]:
## The 0-th row of the matrix-matrix product XW
Z = torch.matmul(X, W)
print(Z)
print(f'0th row of Z:\n {Z[0, :]}')
print(torch.matmul(X[0, :], W))

tensor([[ 16.250, 113.570,   7.330],
        [ 47.200, 114.300,  -4.600],
        [  6.150, 111.900, -18.550],
        [ 41.800, 128.200,  -8.400],
        [ 31.550, 126.520, -26.970],
        [ 47.400, 108.480,  -1.280]])
0th row of Z:
 tensor([ 16.250, 113.570,   7.330])
tensor([ 16.250, 113.570,   7.330])


---

The softmax function: takes a $k$-vector $\mathbf{z}$ as input and returns a vector $\mathbf{a}$ of the same shape as the output which is referred to as the softmax-activated scores.

$\begin{align*}\mathbf{a}&=\text{softmax}(\mathbf{z})=\begin{bmatrix}\dfrac{e^{z_1}}{e^{z_1}+e^{z_2}+\cdots+e^{z_k}}\\\dfrac{e^{z_2}}{e^{z_1}+e^{z_2}+\cdots+e^{z_k}}\\\vdots\\\dfrac{e^{z_k}}{e^{z_1}+e^{z_2}+\cdots+e^{z_k}}\end{bmatrix}.\end{align*}$

In the following example, we consider a raw scores vector $\mathbf{z}$ with 3 components which leads to the softmax-activated scores vectors $\mathbf{a}$ which can be interpreted as the predicted probabilities that the sample belongs to each one of the output classes:

![softmax](https://1drv.ms/i/s!AjTcbXuSD3I3hscmdol7J2G4GDo5WQ?embed=1&width=660)


---

In [ ]:
# Activated raw scores a = softmax(z)
# Raw scores (cooked up)
z = torch.tensor([5.6, 6.4, -4.6])
z = torch.tensor([100, 200, 300])
print(f'Raw scores: \n {z}')
# Exponentiate the raw scores
z_exp = torch.exp(z)
print(f'Exponentiated raw scores: \n {z_exp}')
# Normalize the exponentiated raw scores
z_exp_sum = torch.sum(z_exp)
a = z_exp / z_exp_sum
print(f'Softmax-activated scores: \n {a}')

# A numerically stable way to implement the softmax function
z = z - torch.max(z) # broacasting
z_exp = torch.exp(z)
z_exp_sum = torch.sum(z_exp)
a = z_exp / z_exp_sum
print(a)

Raw scores: 
 tensor([ 5.600,  6.400, -4.600])
Exponentiated raw scores: 
 tensor([  270.426,   601.845,     0.010])
Softmax-activated scores: 
 tensor([    0.310,     0.690,     0.000])
tensor([    0.310,     0.690,     0.000])


---

A numerically stable way to implement the softmax function:

$$\begin{align*}\mathbf{a} &= \text{softmax}(\mathbf{z})\\&=\text{softmax}\left(\begin{bmatrix}5.6\\\color{magenta}{6.4}\\-4.6\end{bmatrix}\right)=\begin{bmatrix}\dfrac{e^{5.6}}{e^{5.6}+e^{\color{magenta}{6.4}}+e^{-4.6}}\\\dfrac{e^{\color{magenta}{6.4}}}{e^{5.6}+e^{\color{magenta}{6.4}}+e^{-4.6}}\\\dfrac{e^{-4.6}}{e^{5.6}+e^{\color{magenta}{6.4}}+e^{-4.6}}\end{bmatrix}\\&=\begin{bmatrix}\dfrac{e^{5.6}\times e^{-\color{magenta}{6.4}}}{\left(e^{5.6}+e^{\color{magenta}{6.4}}+e^{-4.6}\right)\times e^{-\color{magenta}{6.4}}}\\\dfrac{e^{\color{magenta}{6.4}}\times e^{-\color{magenta}{6.4}}}{\left(e^{5.6}+e^{\color{magenta}{6.4}}+e^{-4.6}\right)\times e^{-\color{magenta}{6.4}}}\\\dfrac{e^{-4.6}\times e^{-\color{magenta}{6.4}}}{\left(e^{5.6}+e^{\color{magenta}{6.4}}+e^{-4.6}\right)\times e^{-\color{magenta}{6.4}}}\end{bmatrix}=\begin{bmatrix}\dfrac{e^{5.6-6.4}}{e^{5.6-6.4}+e^{6.4-6.4}+e^{-4.6-6.4}}\\\dfrac{e^{6.4-6.4}}{e^{5.6-6.4}+e^{6.4-6.4}+e^{-4.6-6.4}}\\\dfrac{e^{-4.6-6.4}}{e^{5.6-6.4}+e^{6.4-6.4}+e^{-4.6-6.4}}\end{bmatrix}\\&=\begin{bmatrix}\dfrac{e^{-0.8}}{e^{-0.8}+e^{0.0}+e^{-11}}\\\dfrac{e^{0.0}}{e^{-0.8}+e^0+e^{-11}}\\\dfrac{e^{11.0}}{e^{-0.8}+e^0+e^{-11}}\end{bmatrix}=\begin{bmatrix}0.31\\0.69\\0\end{bmatrix}.\end{align*}$$

---

In [ ]:
# A numerically stable way to implement the softmax function
z = torch.tensor([100, 200, 300])
print(z)
# Subtract the maximum raw score from all the other raw scores
z = z - torch.max(z) # comment this line and see what happens
z_exp = torch.exp(z)
z_exp_sum = torch.sum(z_exp)
a = z_exp / z_exp_sum
print(a)

tensor([100, 200, 300])
tensor([    0.000,     0.000,     1.000])


---

In-built functions for calcuating softmax in PyTorch

---

In [ ]:
## In-built softmax function in PyTorch (dim = 1 corresponds to row-by-row)
## applied to the toy patient data matrix
# Raw scores (cooked up)
z = torch.tensor([5.6, 6.4, -4.6])

# Approach-1 in PyTorch
# Softmax-activated scores
#a = F.softmax(z) # F is an alias for torch.nn.functional
#print(a)

# Approach-2 in PyTorch
#mysoftmax = torch.nn.Softmax(z)
#print(mysoftmax)

# Going back to the raw scores matrix from the patient data
print(f'Raw scores matrix:\n {Z}')

# Softmax-activated scores using approach-1 in PyTorch
A = F.softmax(Z, dim = 1)
print(f'Softmax-activated scores matrix using approach-1:\n {A}')

# Softmax-activated scores using approach-2 in PyTorch
mysoftmax = torch.nn.Softmax(dim = 1) # a function that we have called to be used later
A = mysoftmax(Z)
print(f'Softmax-activated scores matrix using approach-2:\n {A}')

Raw scores matrix:
 tensor([[ 16.250, 113.570,   7.330],
        [ 47.200, 114.300,  -4.600],
        [  6.150, 111.900, -18.550],
        [ 41.800, 128.200,  -8.400],
        [ 31.550, 126.520, -26.970],
        [ 47.400, 108.480,  -1.280]])
Softmax-activated scores matrix using approach-1:
 tensor([[    0.000,     1.000,     0.000],
        [    0.000,     1.000,     0.000],
        [    0.000,     1.000,     0.000],
        [    0.000,     1.000,     0.000],
        [    0.000,     1.000,     0.000],
        [    0.000,     1.000,     0.000]])
Softmax-activated scores matrix using approach-2:
 tensor([[    0.000,     1.000,     0.000],
        [    0.000,     1.000,     0.000],
        [    0.000,     1.000,     0.000],
        [    0.000,     1.000,     0.000],
        [    0.000,     1.000,     0.000],
        [    0.000,     1.000,     0.000]])


---

Setting up the patient data matrix, true output labels, and the initial weights matrix for the softmax classifier:

![data for softmax](https://1drv.ms/i/s!AjTcbXuSD3I3hspfrgklysOtJMOjaA?embed=1&width=800)

---

----

One-hot encoding of the true output labels

---

In [ ]:
# One-hot encoding of the true output labels
y = np.array(['non-diabetic',
              'diabetic',
              'non-diabetic',
              'pre-diabetic',
              'diabetic',
              'pre-diabetic'])

y = y.reshape(-1, 1)
print(y)

ohe = OneHotEncoder(sparse_output = False)
Y = ohe.fit_transform(y)
print(Y)

[['non-diabetic']
 ['diabetic']
 ['non-diabetic']
 ['pre-diabetic']
 ['diabetic']
 ['pre-diabetic']]
[[0. 1. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]
 [0. 0. 1.]]


In [ ]:
# Create the data matrix (read from a file typically)
X = np.array([[72, 120, 37.3, 104, 32.5],
              [85, 130, 37.0, 110, 14],
              [68, 110, 38.5, 125, 34],
              [90, 140, 38.0, 130, 26],
              [84, 132, 38.3, 146, 30],
              [78, 128, 37.2, 102, 12]])
print(f'Data matrix: \n{X}')

# Standardize the data matrix
sc = StandardScaler()
X_S = sc.fit_transform(X)
print(f'Standardized data matrix: \n{X_S}')

# Convert to a PyTorch tensor with the standardized values for the data
X_S = torch.tensor(X_S, dtype = torch.float32)

# Get the number of samples and features
num_samples, num_features = X_S.shape

# Create the output labels vector (also read from a file typically)
y = np.array(['non-diabetic',
              'diabetic',
              'non-diabetic',
              'pre-diabetic',
              'diabetic',
              'pre-diabetic'])

# One-hot encoding of output labels using scikit-learn
ohe = OneHotEncoder(sparse_output = False)
Y = ohe.fit_transform(y.reshape(-1, 1))

# Convert to a PyTorch tensor

# Get the number of labels

# Create the weights matrix



Data matrix: 
[[ 72.  120.   37.3 104.   32.5]
 [ 85.  130.   37.  110.   14. ]
 [ 68.  110.   38.5 125.   34. ]
 [ 90.  140.   38.  130.   26. ]
 [ 84.  132.   38.3 146.   30. ]
 [ 78.  128.   37.2 102.   12. ]]
Standardized data matrix: 
[[-0.979883   -0.70186241 -0.72380201 -0.98707429  0.89204786]
 [ 0.71858087  0.3509312  -1.24493946 -0.60498101 -1.2373567 ]
 [-1.50248727 -1.75465602  1.36074779  0.35025217  1.06470228]
 [ 1.3718362   1.40372481  0.49218537  0.66866323  0.14387869]
 [ 0.5879298   0.56148993  1.01332282  1.68757862  0.60429048]
 [-0.1959766   0.14037248 -0.8975145  -1.11443871 -1.4675626 ]]
